# Creating xG Features

This notebook loads the Statsbomb Indian Super League 2021/2022 360 data,
engineers all shot features, and saves the result to `isli_shots.csv`
for use in the modelling notebook.

In [ ]:
#importing necessary libraries
from mplsoccer import Sbopen
import pandas as pd
import numpy as np
import warnings
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import os
import random as rn
import tensorflow as tf

pd.options.mode.chained_assignment = None
warnings.filterwarnings('ignore')

os.environ['PYTHONHASHSEED'] = '0'
os.environ['CUDA_VISIBLE_DEVICES'] = ''
np.random.seed(1)
rn.seed(1)
tf.random.set_seed(1)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

## Opening data
We load all ISL 2021/2022 matches, extract shots and 360 tracking data,
convert coordinates from yards to metres, and keep only open-play shots
where a goalkeeper was tracked.

In [ ]:
parser = Sbopen()
df_match = parser.match(competition_id=1238, season_id=108)
matches = df_match.match_id.unique()

shot_df = pd.DataFrame()
track_df = pd.DataFrame()

for match in matches:
    df_event = parser.event(match)[0]
    df_track = parser.event(match)[2]
    shots = df_event.loc[df_event['type_name'] == 'Shot']
    shots.x = shots.x.apply(lambda cell: cell*105/120)
    shots.y = shots.y.apply(lambda cell: cell*68/80)
    df_track.x = df_track.x.apply(lambda cell: cell*105/120)
    df_track.y = df_track.y.apply(lambda cell: cell*68/80)
    shot_df = pd.concat([shot_df, shots], ignore_index=True)
    track_df = pd.concat([track_df, df_track], ignore_index=True)

shot_df.reset_index(drop=True, inplace=True)
track_df.reset_index(drop=True, inplace=True)
shot_df = shot_df.loc[shot_df['sub_type_name'] == 'Open Play']
gks_tracked = track_df.loc[track_df['teammate'] == False].loc[track_df['position_name'] == 'Goalkeeper']['id'].unique()
shot_df = shot_df.loc[shot_df['id'].isin(gks_tracked)]

## Feature engineering
We create the ten model features: ball position, shot geometry, goalkeeper
distances, opponent pressure variables, and a basic logistic xG estimate.

In [ ]:
model_vars = shot_df[['id', 'index', 'x', 'y']]
model_vars['goal'] = shot_df.outcome_name.apply(lambda cell: 1 if cell == 'Goal' else 0)
model_vars['goal_smf'] = model_vars['goal'].astype(object)
model_vars['x0'] = model_vars.x
model_vars['x'] = model_vars.x.apply(lambda cell: 105 - cell)
model_vars['c'] = model_vars.y.apply(lambda cell: abs(34 - cell))
model_vars['angle'] = np.where(
    np.arctan(7.32 * model_vars['x'] / (model_vars['x']**2 + model_vars['c']**2 - (7.32/2)**2)) >= 0,
    np.arctan(7.32 * model_vars['x'] / (model_vars['x']**2 + model_vars['c']**2 - (7.32/2)**2)),
    np.arctan(7.32 * model_vars['x'] / (model_vars['x']**2 + model_vars['c']**2 - (7.32/2)**2)) + np.pi
) * 180 / np.pi
model_vars['distance'] = np.sqrt(model_vars['x']**2 + model_vars['c']**2)

# Basic xG via logistic regression (angle + distance only)
def params(df):
    test_model = smf.glm(formula='goal_smf ~ angle + distance', data=df,
                         family=sm.families.Binomial()).fit()
    return test_model.params

def calculate_xG(sh, b):
    bsum = b[0]
    for i, v in enumerate(['angle', 'distance']):
        bsum = bsum + b[i+1] * sh[v]
    return 1 / (1 + np.exp(bsum))

b = params(model_vars)
model_vars['xg_basic'] = model_vars.apply(calculate_xG, b=b, axis=1)

def dist_to_gk(test_shot, track_df):
    gk_pos = track_df.loc[track_df['id'] == test_shot['id']].loc[track_df['teammate'] == False].loc[track_df['position_name'] == 'Goalkeeper'][['x', 'y']]
    return np.sqrt((test_shot['x'] - gk_pos['x'])**2 + (test_shot['y'] - gk_pos['y'])**2).iloc[0]

def y_to_gk(test_shot, track_df):
    gk_pos = track_df.loc[track_df['id'] == test_shot['id']].loc[track_df['teammate'] == False].loc[track_df['position_name'] == 'Goalkeeper'][['y']]
    return abs(test_shot['y'] - gk_pos['y']).iloc[0]

def three_meters_away(test_shot, track_df):
    pp = track_df.loc[track_df['id'] == test_shot['id']].loc[track_df['teammate'] == False][['x', 'y']]
    dist = np.sqrt((test_shot['x'] - pp['x'])**2 + (test_shot['y'] - pp['y'])**2)
    return len(dist[dist < 3])

def players_in_triangle(test_shot, track_df):
    pp = track_df.loc[track_df['id'] == test_shot['id']].loc[track_df['teammate'] == False][['x', 'y']]
    x1, y1 = 105, 34 - 7.32/2
    x2, y2 = 105, 34 + 7.32/2
    x3, y3 = test_shot['x'], test_shot['y']
    xp, yp = pp['x'], pp['y']
    c1 = (x2-x1)*(yp-y1) - (y2-y1)*(xp-x1)
    c2 = (x3-x2)*(yp-y2) - (y3-y2)*(xp-x2)
    c3 = (x1-x3)*(yp-y3) - (y1-y3)*(xp-x3)
    return len(pp.loc[((c1<0) & (c2<0) & (c3<0)) | ((c1>0) & (c2>0) & (c3>0))])

def gk_dist_to_goal(test_shot, track_df):
    gk_pos = track_df.loc[track_df['id'] == test_shot['id']].loc[track_df['teammate'] == False].loc[track_df['position_name'] == 'Goalkeeper'][['x', 'y']]
    return np.sqrt((105 - gk_pos['x'])**2 + (34 - gk_pos['y'])**2).iloc[0]

model_vars['gk_distance']    = shot_df.apply(dist_to_gk,         track_df=track_df, axis=1)
model_vars['gk_distance_y']  = shot_df.apply(y_to_gk,            track_df=track_df, axis=1)
model_vars['close_players']  = shot_df.apply(three_meters_away,   track_df=track_df, axis=1)
model_vars['triangle']       = shot_df.apply(players_in_triangle, track_df=track_df, axis=1)
model_vars['gk_dist_to_goal']= shot_df.apply(gk_dist_to_goal,     track_df=track_df, axis=1)
model_vars['is_closer']      = np.where(model_vars['gk_dist_to_goal'] > model_vars['distance'], 1, 0)
model_vars['header']         = shot_df.body_part_name.apply(lambda cell: 1 if cell == 'Head' else 0)

model_vars.to_csv('isli_shots.csv', index=False)
print(f'Saved isli_shots.csv with {len(model_vars)} shots.')